In [ ]:
#by LLZ
#on 14/10/2021

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import tabula
from time import sleep
import os

print("BO ASFI Web Scraping Tool v.1.0")

#Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'BO ASFI SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

#Assigning the folders that are going to be used in the process
scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder,'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder}
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

#always in list of entities:
regdict = {'BO ASFI 1': 'https://www.asfi.gob.bo/images/INT_FINANCIERA/DOCS/Entidades_supervisadas/Con_Licencia/Con_licencia_Intermediacion.pdf'}

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')

for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(4)
    tempfile = list()
    for times in range(200):
        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
            print('Waiting for file to download')
            sleep(2)
        else:
            break
    else:
        Exception('Failed to download PDF file')
    pdf_file = os.path.join(tempfolder,os.listdir(tempfolder)[0])
    #cam_tabs = camelot.read_pdf(pdf_file, pages='1-end')
    cam_tabs = df = tabula.read_pdf(pdf_file, multiple_tables=True, pages ="all", panda_options={'dtype':str})
    os.remove(pdf_file)
    for table in cam_tabs:
        #print(table.df.columns)
        #print(table.df.head())
        print(table.columns)
        #col0=[ele.replace('\n', ' ').replace('  ', ' ').replace('  ', ' ') for ele in table.df[0].tolist()]
        #col1=[ele.replace('\n', ' ').replace('  ', ' ').replace('  ', ' ') for ele in table.df[1].tolist()]
        col0=[str(ele).replace('\n', ' ').replace('  ', ' ').replace('  ', ' ').strip() for ele in table[0].tolist()]
        col1=[str(ele).replace('\n', ' ').replace('  ', ' ').replace('  ', ' ').strip() for ele in table[1].tolist()]
        col2=[str(ele).replace('\n', ' ').replace('  ', ' ').replace('  ', ' ').strip() for ele in table[1].tolist()]
        for row in range(len(col0)):
            split_name = False
            if any([True if ele.isdigit() else False for ele in col0[row]]):# and '\n' not in col0[row]:
                name = col1[row]
                city = ''
                for etype in ['IFD', 'S.A.', 'R.L.', 'Ltda.']:
                    if etype in name and not name.endswith(etype):
                        part1 = name.split(etype)[0].strip()
                        part2 = name.split(etype)[1].strip()
                        if '(' in part2:
                            name = part1 + f' {etype}' 
                        else:
                            name = part1 + f' {etype}'
                            city = part2
                        break
                else:
                    if len(col2[row])>1:
                        city = col2[row]
                if name in city:
                    city = city.replace(name, '').strip()
                if name.lower()=='la paz':
                    continue
                sqldict['Name'].append(name)
                sqldict['City'].append(city)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['Cntry'].append('BO')
                sqldict['RegCtry'].append('BO')
                sqldict['RegCode'].append('ASFI')
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict['RegulationType'].append("Regulated")
                for key in sqldict.keys():
                    while len(sqldict[key])<len(sqldict['ListProcessDate']):
                        sqldict[key].append('')

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)
sleep(3)

    
    